# 02 · Exploratory Data Analysis
**Ninja Van Capstone 2026 — Last-Mile Delivery Failure Prediction**

This notebook explores the LaDe dataset to understand:
- Class imbalance (failure rate)
- Key feature distributions
- Temporal patterns in delivery failure
- Missing value landscape
- Courier workload distribution

All plots are saved to `/content/outputs/eda/`.

> **Run order:** Run after `01_data_download.ipynb`.


## 1 · Setup & Load Data

In [ ]:
import sys
from pathlib import Path

REPO_DIR = Path("/content/nv-lastmile-training")
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import missingno as msno
import warnings
warnings.filterwarnings("ignore")

RAW_DIR = Path("/content/data/raw")
OUT_DIR = Path("/content/outputs/eda")
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "DejaVu Sans",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})
NV_RED = "#E63946"
NV_GRAY = "#457B9D"

In [ ]:
# Load LaDe from parquet files
parquet_files = sorted((RAW_DIR / "lade").glob("*.parquet"))
if not parquet_files:
    raise FileNotFoundError("No LaDe parquet files found. Run 01_data_download.ipynb first.")

dfs = [pd.read_parquet(f) for f in parquet_files]
df = pd.concat(dfs, ignore_index=True)
print(f"✅ LaDe loaded: {len(df):,} rows × {len(df.columns)} columns")
print(f"   Columns: {list(df.columns)}")

In [ ]:
# Derive delivery_failed target from available columns
# LaDe uses different field names depending on the subset; try in order
def derive_target(df):
    if "delivery_status" in df.columns:
        s = df["delivery_status"].astype(str).str.upper().str.strip()
        failed  = s.isin(["FAILED", "RETURNED", "EXCEPTION", "REFUSED"])
        success = s.isin(["DELIVERED", "SUCCESS", "COMPLETED"])
        result  = pd.Series(np.nan, index=df.index)
        result[failed]  = 1
        result[success] = 0
        return result
    if "finish_time" in df.columns:
        return df["finish_time"].isna().astype(float)
    if "got_time" in df.columns and "finish_time" in df.columns:
        return df["finish_time"].isna().astype(float)
    return pd.Series(np.nan, index=df.index)

df["delivery_failed"] = derive_target(df)
df_labeled = df.dropna(subset=["delivery_failed"]).copy()
df_labeled["delivery_failed"] = df_labeled["delivery_failed"].astype(int)

print(f"Labeled rows: {len(df_labeled):,} (dropped {len(df) - len(df_labeled):,} unknown outcomes)")
print(f"Failure rate: {df_labeled['delivery_failed'].mean():.3f}")

## 2 · Class Imbalance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: overall counts
counts = df_labeled["delivery_failed"].value_counts().sort_index()
bars = axes[0].bar(["Success (0)", "Failure (1)"], counts.values,
                    color=[NV_GRAY, NV_RED], edgecolor="white", linewidth=1.5)
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + counts.max()*0.01,
                 f"{val:,}\n({val/len(df_labeled)*100:.1f}%)",
                 ha="center", va="bottom", fontsize=10)
axes[0].set_title("Class Distribution", fontweight="bold")
axes[0].set_ylabel("Count")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

# Right: donut chart
wedges, texts, autotexts = axes[1].pie(
    counts.values,
    labels=["Success", "Failure"],
    colors=[NV_GRAY, NV_RED],
    autopct="%1.1f%%",
    startangle=90,
    wedgeprops={"width": 0.5, "edgecolor": "white", "linewidth": 2},
)
for at in autotexts:
    at.set_fontsize(11)
axes[1].set_title("Failure vs Success Split", fontweight="bold")

plt.suptitle("LaDe Dataset — Target Class Distribution", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
fig.savefig(OUT_DIR / "01_class_distribution.png", bbox_inches="tight", dpi=150)
plt.show()
print(f"\n  Total samples : {len(df_labeled):,}")
print(f"  Failures      : {counts.get(1, 0):,}  ({counts.get(1, 0)/len(df_labeled)*100:.2f}%)")
print(f"  Successes     : {counts.get(0, 0):,}  ({counts.get(0, 0)/len(df_labeled)*100:.2f}%)")
print(f"  Imbalance ratio (neg:pos): {counts.get(0, 0)/max(counts.get(1, 1), 1):.2f}:1")

## 3 · Failure Rate by City

In [ ]:
city_col = next((c for c in ["city", "city_id", "city_code"] if c in df_labeled.columns), None)

if city_col:
    city_stats = (
        df_labeled.groupby(city_col)["delivery_failed"]
        .agg(failure_rate="mean", count="count")
        .reset_index()
        .sort_values("failure_rate", ascending=True)
    )

    fig, ax = plt.subplots(figsize=(10, max(4, len(city_stats) * 0.45)))
    bars = ax.barh(city_stats[city_col].astype(str), city_stats["failure_rate"] * 100,
                   color=NV_RED, alpha=0.82, edgecolor="white")

    for bar, row in zip(bars, city_stats.itertuples()):
        ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
                f"{row.failure_rate*100:.1f}%  (n={row.count:,})",
                va="center", fontsize=9)

    ax.axvline(df_labeled["delivery_failed"].mean() * 100,
               color="black", linestyle="--", lw=1.2, label=f"Overall avg: {df_labeled['delivery_failed'].mean()*100:.1f}%")
    ax.set_xlabel("Failure Rate (%)")
    ax.set_title("Delivery Failure Rate by City", fontweight="bold")
    ax.legend()
    plt.tight_layout()
    fig.savefig(OUT_DIR / "02_failure_rate_by_city.png", bbox_inches="tight", dpi=150)
    plt.show()
else:
    print("⚠️  No city column found — skipping city breakdown")

## 4 · Temporal Patterns

In [ ]:
# Parse timestamps
for col in ["start_time", "got_time", "finish_time"]:
    if col in df_labeled.columns:
        df_labeled[col] = pd.to_datetime(df_labeled[col], errors="coerce")

ref_col = next((c for c in ["start_time", "got_time"] if c in df_labeled.columns), None)

if ref_col:
    df_labeled["hour_of_day"] = df_labeled[ref_col].dt.hour
    df_labeled["day_of_week"] = df_labeled[ref_col].dt.dayofweek
    df_labeled["day_name"]    = df_labeled[ref_col].dt.day_name()
    print(f"✅ Parsed timestamps from '{ref_col}'")
else:
    # Synthesise from ds (date string)
    if "ds" in df_labeled.columns:
        df_labeled["_dt"]      = pd.to_datetime(df_labeled["ds"], errors="coerce")
        df_labeled["hour_of_day"] = 12  # unknown — use midday
        df_labeled["day_of_week"] = df_labeled["_dt"].dt.dayofweek
        df_labeled["day_name"]    = df_labeled["_dt"].dt.day_name()
        print("ℹ️  No timestamp column — derived day_of_week from 'ds' date string")
    else:
        print("⚠️  No temporal columns available for time-based EDA")
        df_labeled["hour_of_day"] = 12
        df_labeled["day_of_week"] = 2
        df_labeled["day_name"]    = "Wednesday"

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Failure rate by hour
hour_stats = (
    df_labeled.groupby("hour_of_day")["delivery_failed"]
    .agg(rate="mean", count="count")
    .reset_index()
)
axes[0].bar(hour_stats["hour_of_day"], hour_stats["rate"] * 100,
            color=NV_RED, alpha=0.82, edgecolor="white", width=0.8)
axes[0].axhline(df_labeled["delivery_failed"].mean() * 100,
                color="black", linestyle="--", lw=1.2,
                label=f"Overall avg: {df_labeled['delivery_failed'].mean()*100:.1f}%")
axes[0].set_xlabel("Hour of Day (24h)")
axes[0].set_ylabel("Failure Rate (%)")
axes[0].set_title("Failure Rate by Hour of Day", fontweight="bold")
axes[0].set_xticks(range(0, 24, 2))
axes[0].legend()

# Failure rate by day of week
day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
dow_stats = (
    df_labeled.groupby("day_name")["delivery_failed"]
    .agg(rate="mean", count="count")
    .reindex(day_order)
    .reset_index()
)
colors = [NV_RED if d in ["Saturday","Sunday"] else NV_GRAY for d in day_order]
axes[1].bar(dow_stats["day_name"], dow_stats["rate"] * 100,
            color=colors, alpha=0.82, edgecolor="white")
axes[1].axhline(df_labeled["delivery_failed"].mean() * 100,
                color="black", linestyle="--", lw=1.2)
axes[1].set_xlabel("Day of Week")
axes[1].set_ylabel("Failure Rate (%)")
axes[1].set_title("Failure Rate by Day of Week", fontweight="bold")
axes[1].tick_params(axis="x", rotation=30)

plt.suptitle("Temporal Failure Patterns", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
fig.savefig(OUT_DIR / "03_temporal_failure_patterns.png", bbox_inches="tight", dpi=150)
plt.show()

## 5 · Delivery Time Window Distribution

In [ ]:
if "start_time" in df_labeled.columns and "end_time" in df_labeled.columns:
    df_labeled["end_time"] = pd.to_datetime(df_labeled["end_time"], errors="coerce")
    df_labeled["window_mins"] = (
        (df_labeled["end_time"] - df_labeled["start_time"])
        .dt.total_seconds() / 60
    ).clip(0, 720)

    valid_windows = df_labeled["window_mins"].dropna()
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Distribution of window sizes
    axes[0].hist(valid_windows, bins=40, color=NV_GRAY, edgecolor="white", alpha=0.85)
    axes[0].axvline(valid_windows.median(), color=NV_RED, linestyle="--",
                    label=f"Median: {valid_windows.median():.0f} min")
    axes[0].set_xlabel("Window Duration (minutes)")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Distribution of Delivery Time Windows", fontweight="bold")
    axes[0].legend()
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

    # Failure rate by window width bucket
    bins = [0, 60, 120, 240, 480, float("inf")]
    labels = ["<1h", "1–2h", "2–4h", "4–8h", ">8h"]
    df_labeled["window_bucket"] = pd.cut(df_labeled["window_mins"], bins=bins, labels=labels)
    wb_stats = (
        df_labeled.groupby("window_bucket")["delivery_failed"]
        .agg(rate="mean", count="count")
        .reset_index()
    )
    axes[1].bar(wb_stats["window_bucket"].astype(str), wb_stats["rate"] * 100,
                color=NV_RED, alpha=0.82, edgecolor="white")
    axes[1].set_xlabel("Window Duration Bucket")
    axes[1].set_ylabel("Failure Rate (%)")
    axes[1].set_title("Failure Rate by Window Width", fontweight="bold")

    plt.suptitle("Delivery Time Window Analysis", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    fig.savefig(OUT_DIR / "04_time_window_distribution.png", bbox_inches="tight", dpi=150)
    plt.show()
else:
    print("⚠️  start_time / end_time columns not found — skipping window analysis")

## 6 · Courier Workload Distribution

In [ ]:
courier_col = next((c for c in ["courier_id","driver_id","employee_id"] if c in df_labeled.columns), None)
date_col    = next((c for c in ["ds","date","delivery_date"]            if c in df_labeled.columns), None)

if courier_col and date_col:
    daily_load = (
        df_labeled.groupby([courier_col, date_col])
        .size()
        .reset_index(name="daily_parcels")
    )

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Histogram of daily loads
    axes[0].hist(daily_load["daily_parcels"], bins=40, color=NV_GRAY,
                 edgecolor="white", alpha=0.85)
    axes[0].axvline(daily_load["daily_parcels"].median(), color=NV_RED,
                    linestyle="--", label=f"Median: {daily_load['daily_parcels'].median():.0f}")
    axes[0].set_xlabel("Parcels per Courier per Day")
    axes[0].set_ylabel("Count")
    axes[0].set_title("Courier Daily Workload Distribution", fontweight="bold")
    axes[0].legend()
    axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

    # Failure rate by load bucket
    df_with_load = df_labeled.merge(daily_load, on=[courier_col, date_col], how="left")
    load_bins   = [0, 10, 25, 40, 60, float("inf")]
    load_labels = ["1–10", "11–25", "26–40", "41–60", ">60"]
    df_with_load["load_bucket"] = pd.cut(
        df_with_load["daily_parcels"], bins=load_bins, labels=load_labels
    )
    load_stats = (
        df_with_load.groupby("load_bucket")["delivery_failed"]
        .agg(rate="mean", count="count")
        .reset_index()
    )
    axes[1].bar(load_stats["load_bucket"].astype(str), load_stats["rate"] * 100,
                color=NV_RED, alpha=0.82, edgecolor="white")
    axes[1].set_xlabel("Daily Parcel Count Bucket")
    axes[1].set_ylabel("Failure Rate (%)")
    axes[1].set_title("Failure Rate by Courier Workload", fontweight="bold")

    plt.suptitle("Courier Workload Analysis", fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout()
    fig.savefig(OUT_DIR / "05_courier_workload.png", bbox_inches="tight", dpi=150)
    plt.show()

    print(f"  Median daily load : {daily_load['daily_parcels'].median():.0f} parcels")
    print(f"  Max daily load    : {daily_load['daily_parcels'].max():.0f} parcels")
    print(f"  Unique couriers   : {df_labeled[courier_col].nunique():,}")
else:
    print("⚠️  courier_id or date column not found — skipping workload analysis")

## 7 · Missing Value Heatmap

In [ ]:
import missingno as msno

# Show missingness on a sample to keep it readable
sample = df.sample(min(5000, len(df)), random_state=42)

fig = plt.figure(figsize=(14, 6))
msno.matrix(sample, color=(0.28, 0.47, 0.62), sparkline=True, fontsize=10, ax=plt.gca())
plt.title("Missing Value Matrix (5,000-row sample)", fontsize=13, fontweight="bold", pad=20)
plt.tight_layout()
fig.savefig(OUT_DIR / "06_missing_values.png", bbox_inches="tight", dpi=150)
plt.show()

# Null percentage table
null_pct = (df.isnull().mean() * 100).sort_values(ascending=False)
print("\nNull Percentages:")
print(null_pct[null_pct > 0].to_frame("null_%").to_string())

## 8 · Correlation Matrix

In [ ]:
numeric_cols = df_labeled.select_dtypes(include=[np.number]).columns.tolist()
# Exclude internal / id columns
exclude = [c for c in numeric_cols if any(k in c.lower() for k in ["id","_split","index"])]
numeric_cols = [c for c in numeric_cols if c not in exclude][:20]  # cap at 20

if numeric_cols:
    corr = df_labeled[numeric_cols].corr()

    fig, ax = plt.subplots(figsize=(max(8, len(numeric_cols) * 0.55),
                                     max(6, len(numeric_cols) * 0.5)))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, cmap="RdBu_r", center=0,
                annot=len(numeric_cols) <= 15,
                fmt=".2f", linewidths=0.5, ax=ax,
                annot_kws={"size": 8},
                vmin=-1, vmax=1)
    ax.set_title("Correlation Matrix — Numeric Features", fontsize=13, fontweight="bold")
    plt.tight_layout()
    fig.savefig(OUT_DIR / "07_correlation_matrix.png", bbox_inches="tight", dpi=150)
    plt.show()
else:
    print("⚠️  No numeric columns available for correlation matrix")

## 9 · Summary Statistics Table

In [ ]:
print("=" * 70)
print("  DATASET SUMMARY — LaDe")
print("=" * 70)

summary_stats = {
    "Total rows":          f"{len(df):,}",
    "Labeled rows":        f"{len(df_labeled):,}",
    "Overall failure rate":f"{df_labeled['delivery_failed'].mean():.3f}",
    "Columns":             str(len(df.columns)),
    "Null rate (overall)": f"{df.isnull().mean().mean()*100:.2f}%",
}

if "courier_id" in df_labeled.columns:
    summary_stats["Unique couriers"] = f"{df_labeled['courier_id'].nunique():,}"
if "ds" in df_labeled.columns:
    summary_stats["Date range"] = (
        f"{df_labeled['ds'].min()} → {df_labeled['ds'].max()}"
    )
if city_col:
    summary_stats["Cities covered"] = f"{df_labeled[city_col].nunique()}"

for k, v in summary_stats.items():
    print(f"  {k:30s}: {v}")

print("\n" + "=" * 70)
print(f"✅ EDA plots saved → {OUT_DIR}")
print("✅ NOTEBOOK 02 COMPLETE — proceed to 03_feature_engineering.ipynb")
print("=" * 70)